In [2]:
import sys
sys.path.append('..')

In [3]:
import tqdm
import torch
import pickle
import warnings
import numpy as np
import pandas as pd
import torch.optim as optim
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import ParameterGrid

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

warnings.filterwarnings('ignore')

In [ ]:
# alphas = (0.02, 0.04, 0.2)      # <------------- Don't Touch
alphas = np.linspace(0, 0.2, 11).round(3)      # <------------- Don't Touch
sba_l1psd_seed = 7              # <------------- Don't Touch
clf = "lr"
datasets = ["synthesis", "sba"]
recourse_fns = [LARRecourse]    # <---------------------
lambdas = [0.1, 0.2, 0.3]       # <-------------
save_file = True
should_sample = False
divide_by_norm = True


for dataset in datasets:
    # Read File and Get results
    # file_path = f"../results/cost_validity_latest/{dataset}_correctRELU.pickle"
    file_path = f"../results/cost_validity_latest/lr_{dataset}_alg1_lamb0.1.pickle"
    
    ret = pd.read_pickle(file_path)

    x0s = np.array(ret['x_0'])
    x0s = x0s[0]
    if dataset == "sba" and should_sample:
        rng = np.random.default_rng(seed=sba_l1psd_seed)
        size_N = int(np.rint(0.15 * x0s.shape[0]))
        x0s = rng.choice(x0s, size=size_N, replace=False)

    try:
        theta0 = ret['theta_0']['out.weight'][0].numpy().astype(np.float64)
        bias0 = ret['theta_0']['out.bias'].numpy().astype(np.float64) 
    except TypeError:
        try: 
            theta0 = ret['theta_0'][0][0][0][0].numpy().astype(np.float64)
            bias0 = ret['theta_0'][0][0][1].numpy().astype(np.float64)
        except IndexError:
            theta0 = ret['theta_0'][0][0][0].numpy().astype(np.float64)
            bias0 = ret['theta_0'][0][1].numpy().astype(np.float64)
    if divide_by_norm:
        divider = np.linalg.norm(theta0, 2)
        theta0 = theta0 / divider
        bias0 = bias0 / divider

    ret['x_0'] = ret['x_0'][:len(alphas)]
    filtered_paramVal = [val['delta_max'] for val in ret['params'] if val['delta_max'] in alphas]
    print(filtered_paramVal)
    ret['params'] = ParameterGrid({'delta_max': filtered_paramVal})

    for recourse_fn in recourse_fns:
        for lamb in lambdas:
            xRs = [] # Final recouuse list

            if dataset == "sba" and should_sample:
                for i in range(len(ret['x_0'])):
                    ret['x_0'][i] = x0s 

            for alpha in alphas:
                res = []
                reco = recourse_fn(weights=theta0, bias=bias0, alpha=alpha, lamb=lamb)

                for x0 in tqdm.tqdm(x0s, desc=f"Running {clf}_{dataset} with lambda = {lamb}, alpha = {alpha}"):
                    res.append(reco.get_recourse(x0))
                xRs.append(res)
            
            ret['x_r'] = xRs
            
            # Save each (dataset, recourse function, lambda) file
            if save_file:
                file_path_saved = f"../results/cost_validity_latest/{clf}_{dataset}_{reco.name}_lamb{lamb}_newRELU.pickle"
                with open(file_path_saved, 'wb') as outFile:
                    pickle.dump(ret, outFile)
                    print(f"{clf}_{dataset}_{reco.name}_lamb{lamb}_newRELU.pickle is saved!")

[np.float64(0.0), np.float64(0.02), np.float64(0.04), np.float64(0.06), np.float64(0.08), np.float64(0.1), np.float64(0.12), np.float64(0.14), np.float64(0.16), np.float64(0.18), np.float64(0.2)]


Running lr_synthesis with lambda = 0.1, alpha = 0.0: 100%|██████████| 100/100 [00:00<00:00, 13139.64it/s]
Running lr_synthesis with lambda = 0.1, alpha = 0.02: 100%|██████████| 100/100 [00:00<00:00, 17462.44it/s]
Running lr_synthesis with lambda = 0.1, alpha = 0.04: 100%|██████████| 100/100 [00:00<00:00, 12580.02it/s]
Running lr_synthesis with lambda = 0.1, alpha = 0.06: 100%|██████████| 100/100 [00:00<00:00, 12935.80it/s]
Running lr_synthesis with lambda = 0.1, alpha = 0.08: 100%|██████████| 100/100 [00:00<00:00, 12148.60it/s]
Running lr_synthesis with lambda = 0.1, alpha = 0.1: 100%|██████████| 100/100 [00:00<00:00, 12208.71it/s]
Running lr_synthesis with lambda = 0.1, alpha = 0.12: 100%|██████████| 100/100 [00:00<00:00, 9383.65it/s]
Running lr_synthesis with lambda = 0.1, alpha = 0.14: 100%|██████████| 100/100 [00:00<00:00, 11621.47it/s]
Running lr_synthesis with lambda = 0.1, alpha = 0.16: 100%|██████████| 100/100 [00:00<00:00, 14252.28it/s]
Running lr_synthesis with lambda = 0.1, 

lr_synthesis_alg1_lamb0.1_newRELU.pickle is saved!


Running lr_synthesis with lambda = 0.2, alpha = 0.0: 100%|██████████| 100/100 [00:00<00:00, 13675.15it/s]
Running lr_synthesis with lambda = 0.2, alpha = 0.02: 100%|██████████| 100/100 [00:00<00:00, 14427.30it/s]
Running lr_synthesis with lambda = 0.2, alpha = 0.04: 100%|██████████| 100/100 [00:00<00:00, 12464.87it/s]
Running lr_synthesis with lambda = 0.2, alpha = 0.06: 100%|██████████| 100/100 [00:00<00:00, 7100.45it/s]
Running lr_synthesis with lambda = 0.2, alpha = 0.08: 100%|██████████| 100/100 [00:00<00:00, 12876.23it/s]
Running lr_synthesis with lambda = 0.2, alpha = 0.1: 100%|██████████| 100/100 [00:00<00:00, 10810.90it/s]
Running lr_synthesis with lambda = 0.2, alpha = 0.12: 100%|██████████| 100/100 [00:00<00:00, 13276.05it/s]
Running lr_synthesis with lambda = 0.2, alpha = 0.14: 100%|██████████| 100/100 [00:00<00:00, 12415.06it/s]
Running lr_synthesis with lambda = 0.2, alpha = 0.16: 100%|██████████| 100/100 [00:00<00:00, 8982.15it/s]
Running lr_synthesis with lambda = 0.2, a

lr_synthesis_alg1_lamb0.2_newRELU.pickle is saved!


...

lr_sba_alg1_lamb0.1_newRELU.pickle is saved!


Running lr_sba with lambda = 0.2, alpha = 0.2: 100%|██████████| 100/100 [00:00<00:00, 4937.44it/s]


lr_sba_alg1_lamb0.2_newRELU.pickle is saved!


Running lr_sba with lambda = 0.3, alpha = 0.2: 100%|██████████| 100/100 [00:00<00:00, 6616.04it/s]

lr_sba_alg1_lamb0.3_newRELU.pickle is saved!
